# 🇮🇳 VoiceGuard India - AWS Pandas Master Pipeline

This pipeline completely bypasses the buggy HuggingFace `datasets` library. We use `hf_hub_download` for lightning-fast downloads, and Pandas to read the raw Parquet files directly, guaranteeing zero JSON errors.

In [ ]:
# 1. INSTALL OS-LEVEL DECODERS
# This installs ffmpeg at the OS level so librosa can decode MP3 and Opus files.
!conda install -y -c conda-forge ffmpeg

In [ ]:
# 2. INSTALL REQUIRED PYTHON LIBRARIES
!pip install pandas librosa xgboost scikit-learn fsspec pyarrow huggingface_hub soundfile

In [ ]:
# 3. DOWNLOAD & EXTRACT FEATURES USING PANDAS
import pandas as pd
import librosa
import numpy as np
import io
import pickle
import xgboost as xgb
from sklearn.metrics import classification_report
from tqdm import tqdm
from huggingface_hub import HfFileSystem, hf_hub_download
import warnings
warnings.filterwarnings('ignore')

# ⚠️ PASTE YOUR HUGGINGFACE TOKEN HERE ⚠️
HF_TOKEN = "hf_YOUR_TOKEN_HERE"

def extract_from_bytes(audio_bytes, sr=16000):
    try:
        y, sr = librosa.load(io.BytesIO(audio_bytes['bytes']), sr=sr, duration=3.0)
        mfccs = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20).T, axis=0)
        centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(y))
        return np.hstack((mfccs, centroid, zcr))
    except Exception as e:
        return None

fs = HfFileSystem(token=HF_TOKEN)

def get_repo_and_file(repo_id, keyword):
    try:
        files = fs.glob(f"datasets/{repo_id}/**/*.parquet")
        for f in files:
            if keyword in f and 'train' in f:
                return f
        for f in files:
            if 'train' in f:
                return f
    except:
        pass
    return None

X, y = [], []
SAMPLES_PER_DATASET = 2000

datasets_to_download = [
    {"name": "Google Fleurs", "label": 1, "repo": "google/fleurs", "kw": "hi_in"},
    {"name": "IndicTTS", "label": 1, "repo": "SPRINGLab/IndicTTS-Hindi", "kw": "train"},
    {"name": "UD-NLP", "label": 1, "repo": "ud-nlp/hindi-speech-recognition-dataset", "kw": "train"},
    {"name": "Sarvam Deepfakes", "label": 0, "repo": "abhirl/hindi-tts-dataset", "kw": "train"}
]

for ds in datasets_to_download:
    print(f"\n🔍 Locating Parquet files for: {ds['name']}...")
    hf_path = get_repo_and_file(ds['repo'], ds['kw'])
    
    if not hf_path:
        print(f"⚠️ Could not find Parquet file in {ds['repo']}")
        continue
        
    parts = hf_path.split("/")
    repo_id = f"{parts[1]}/{parts[2]}"
    filename = "/".join(parts[3:])
    
    print(f"🌍 Fast-Downloading {filename} (Watch the progress bar!)...")
    try:
        # Natively download the file super fast with a progress bar!
        local_path = hf_hub_download(repo_id=repo_id, filename=filename, repo_type="dataset", token=HF_TOKEN)
        
        print(f"⏬ Extracting Features from local file...")
        df = pd.read_parquet(local_path)
        
        count = 0
        for _, row in tqdm(df.iterrows(), total=len(df)):
            if count >= SAMPLES_PER_DATASET: break
            if 'audio' in row and row['audio'] is not None:
                features = extract_from_bytes(row['audio'])
                if features is not None:
                    X.append(features)
                    y.append(ds['label'])
                    count += 1
    except Exception as e:
        print(f"⚠️ Error with {ds['name']}: {e}")

X_final = np.array(X)
y_final = np.array(y)

if len(np.unique(y_final)) < 2:
    print("\n\n❌ FATAL ERROR: Missing Authentic or Deepfake data!")
else:
    print(f"\n✅ SUCCESS! Extracted {len(X_final)} files! Ready to train XGBoost!")

In [ ]:
# 4. TRAIN XGBOOST DEEPFAKE MODEL
if len(np.unique(y_final)) >= 2:
    print("\n🚀 Training XGBoost Classifier...")
    clf = xgb.XGBClassifier(n_estimators=300, max_depth=10, learning_rate=0.05, n_jobs=-1, random_state=42)
    clf.fit(X_final, y_final)
    
    print("\n✅ Training Complete! Accuracy Metrics:")
    predictions = clf.predict(X_final)
    print(classification_report(y_final, predictions, target_names=["Deepfake", "Authentic"]))
    
    with open("voiceguard_ultimate_model.pkl", "wb") as f:
        pickle.dump(clf, f)
        
    print("\n💾 SUCCESS! Ultimate Model saved as 'voiceguard_ultimate_model.pkl'")